# Learning from Quantum Algorithms
Chapter 8 companion. Input X is the high-order register and target Y=q0 is the low-order bit. One-input ket order is |q1 q0⟩, two-input order is |q2 q1 q0⟩. Statevectors below are ideal model information.


In [ ]:
%pip -q install "qiskit>=2,<3" "numpy>=2,<3"


In [ ]:
import numpy as np
from itertools import product, combinations
from math import comb
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
z=np.array([1,0],complex)
o=np.array([0,1],complex)
minus=np.array([1,-1],complex)/np.sqrt(2)
H=np.array([[1,1],[1,-1]],complex)/np.sqrt(2)
def classify(f):
    r=sum(f)
    return "constant" if r in [0,len(f)] else "balanced" if r==len(f)//2 else "outside promise"
def signed_amplitudes(f):
    N=len(f)
    return np.array([sum((-1)**(f[x]+((x&out).bit_count()%2)) for x in range(N))/N for out in range(N)])
def oracle_matrix(f):
    U=np.zeros((2*len(f),2*len(f)),complex)
    for x,fx in enumerate(f):
        for y in [0,1]:
            U[2*x+(y^fx),2*x+y]=1
    return U
def small_oracle(f):
    n=int(np.log2(len(f)))
    assert n in [1,2]
    qc=QuantumCircuit(n+1,name="Uf")
    for x,fx in enumerate(f):
        if fx:
            zero_controls=[j+1 for j in range(n) if not ((x>>j)&1)]
            for j in zero_controls:
                qc.x(j)
            if n==1:
                qc.cx(1,0)
            else:
                qc.ccx(1,2,0)
            for j in reversed(zero_controls):
                qc.x(j)
    return qc


## 1. One classical response leaves two classes compatible
List functions compatible with each possible response. The model truth table is not supplied to the algorithm.


In [ ]:
tables=list(product([0,1],repeat=2))
for x in [0,1]:
    for response in [0,1]:
        compatible=[f for f in tables if f[x]==response]
        assert {classify(f) for f in compatible}=={"constant","balanced"}
        print("Query",x,"response",response,"compatible",compatible)


## 2. Deutsch: full state sequence and exact vector phases
The input is q1 and target q0. Compare each circuit stage with an independent joint-column calculation.


In [ ]:
for f in tables:
    qc=QuantumCircuit(2)
    qc.x(0)
    assert np.allclose(Statevector.from_instruction(qc).data,np.kron(z,o))
    qc.h(1)
    qc.h(0)
    prepared=np.kron(np.array([1,1])/np.sqrt(2),minus)
    assert np.allclose(Statevector.from_instruction(qc).data,prepared)
    U=oracle_matrix(f)
    assert np.allclose(Operator(small_oracle(f)).data,U)
    qc.compose(small_oracle(f),inplace=True)
    post=np.kron(np.array([(-1)**f[0],(-1)**f[1]])/np.sqrt(2),minus)
    assert np.allclose(Statevector.from_instruction(qc).data,post)
    qc.h(1)
    final=Statevector.from_instruction(qc)
    expected=((-1)**f[0])*np.kron(z if f[0]==f[1] else o,minus)
    assert np.allclose(final.data,expected)
    expected_probs=[1,0] if f[0]==f[1] else [0,1]
    assert np.allclose(final.probabilities([1]),expected_probs)
    print(f,classify(f),final.probabilities([1]))


## 3. Interference versus an intermediate measurement
Averaging the measured input branches before the final H gives a uniform final distribution for every table. This is a different procedure from the coherent circuit.


In [ ]:
for f in tables:
    coherent=np.abs(signed_amplitudes(f))**2
    measured=.5*np.abs(H@z)**2+.5*np.abs(H@o)**2
    assert np.allclose(measured,[.5,.5])
    assert np.allclose(coherent,[1,0] if f[0]==f[1] else [0,1])


## 4. Two-input Deutsch-Jozsa, including invalid-promise cases
Test all sixteen tables, not just selected examples. The all-zero probability is meaningful mathematically for every table, but the class guarantee requires the promise.


In [ ]:
for f in product([0,1],repeat=4):
    qc=QuantumCircuit(3)
    qc.x(0)
    for j in [0,1,2]:
        qc.h(j)
    qc.compose(small_oracle(f),inplace=True)
    qc.h(1)
    qc.h(2)
    final=Statevector.from_instruction(qc)
    amplitudes=signed_amplitudes(f)
    expected=np.kron(amplitudes,minus)
    assert np.allclose(final.data,expected)
    p=np.abs(final.data.reshape(4,2))**2
    p=p.sum(axis=1)
    assert np.allclose(p,np.abs(amplitudes)**2)
    assert np.isclose(p[0],(1-2*sum(f)/4)**2)
    if classify(f)=="constant":
        assert np.isclose(p[0],1)
    elif classify(f)=="balanced":
        assert np.isclose(p[0],0)
        assert np.count_nonzero(p>1e-10)==1 # special to two input bits
    if f in [(0,1,0,1),(0,0,0,1)]:
        print(f,classify(f),p)


## 5. A balanced three-input function with several possible outputs
Majority f(x2,x1,x0) is one when at least two input bits are one. It is balanced but not affine. Its reversible oracle is three pairwise-controlled target flips: xy XOR xz XOR yz.


In [ ]:
f=[int(x.bit_count()>=2) for x in range(8)]
assert classify(f)=="balanced"
majority=QuantumCircuit(4,name="majority")
majority.ccx(1,2,0)
majority.ccx(1,3,0)
majority.ccx(2,3,0)
assert np.allclose(Operator(majority).data,oracle_matrix(f))
qc=QuantumCircuit(4)
qc.x(0)
for j in range(4):
    qc.h(j)
qc.compose(majority,inplace=True)
for j in [1,2,3]:
    qc.h(j)
sv=Statevector.from_instruction(qc)
a=signed_amplitudes(f)
assert np.allclose(sv.data,np.kron(a,minus))
assert np.isclose(abs(a[0])**2,0)
assert np.count_nonzero(abs(a)>1e-10)>1
print("Input distribution:",{format(i,"03b"):float(abs(v)**2) for i,v in enumerate(a) if abs(v)>1e-10})
sv.seed(2026)
saved=sv.data.copy()
print("200 fresh input-register shots:",sv.sample_counts(200,qargs=[1,2,3]))
assert np.allclose(sv.data,saved)


## 6. Classical success criteria
For k independent uniform queries with replacement, a fixed balanced function is misclassified only when all outputs agree. This error is 2^(1-k). It is not a posterior probability over functions.


In [ ]:
f=[0,0,1,1]
N=len(f)
for k in range(1,5):
    sequences=list(product(range(N),repeat=k))
    error=sum(len({f[x] for x in seq})==1 for seq in sequences)/len(sequences)
    assert np.isclose(error,2**(1-k))
for k in range(1,N+1):
    subsets=list(combinations(range(N),k))
    error=sum(len({f[x] for x in seq})==1 for seq in subsets)/len(subsets)
    expected=2*comb(N//2,k)/comb(N,k) if k<=N//2 else 0
    assert np.isclose(error,expected)
n=5
k=8
print("n=5 exact classical worst-case queries:",2**(n-1)+1)
print("Ideal quantum oracle queries:",1)
print("k=8 balanced error for independent classical sampling:",2**(1-k))


## Practice
1. Derive the decision bit for all four one-bit truth tables, retaining the global sign until the final probability calculation.
2. Replace the two-input table by AND and explain why a zero record does not imply constant without the promise.
3. Explain why the three-input majority example never produces zero but does not concentrate on one nonzero label.
4. Compare exact query guarantees with an implementation on a noisy device. List costs omitted by oracle-query counting.

Further reading: [Deutsch](https://qiskit.qotlabs.org/learning/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm), [Deutsch-Jozsa](https://qiskit.qotlabs.org/learning/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-jozsa-algorithm).
